In [ ]:
from math import ceil
import pulp as pl
import pandas as pd
from IPython.display import display, Markdown
from pathlib import Path

In [ ]:
import importlib
import utils.util_display
import utils.util_planning_input_validation

importlib.reload(utils.util_display)
from utils.util_display import *

importlib.reload(utils.util_planning_input_validation)
from utils.util_planning_input_validation import *

In [ ]:
excel_file = Path("xlconfigs") / "material_input.xlsm"

# Import Master Lists: Products and Materials
df_master = pd.read_excel(excel_file, sheet_name="Master_Lists")

products, materials = [
    df_master[col].dropna().tolist()
    for col in ["Product_List", "Material_List"]
]

# Fixed production plan: (product, due day) -> production quantity
df_sched = pd.read_excel(excel_file, sheet_name="Production_Schedule", skiprows=2)
df_sched = df_sched.set_index("Product")

num_days = len(df_sched.columns)
days = range(1, num_days + 1)

production_due = {}
for prod in products:
    for d in days:
        val = df_sched.iloc[
            products.index(prod), d - 1
        ]  # Column index starts at 0
        if pd.notna(val) and val > 0:
            production_due[(prod, d)] = float(val)

# Material required for one finished-product unit
df_bom = pd.read_excel(excel_file, sheet_name="BOM_Matrix", index_col=0)
bom = {
    (p, m): float(df_bom.loc[m, p]) for p in products for m in materials
}

# Convert the production plan into daily material requirements
requirement = {
    (m, d): sum(
        production_due.get((p, d), 0) * bom[p, m]
        for p in products
    )
    for m in materials
    for d in days
}

**Config**

In [ ]:
df_config = pd.read_excel(
    excel_file, sheet_name="Material_Config", index_col=0
)

warehouse_capacity = float(df_config.iloc[0, 10])

# Lead_Time forced to integer for day indexing
lead_time = df_config["Lead_Time"].astype(int).to_dict()

# All others safely converted to floats
initial_inventory = df_config["Initial_Inventory"].astype(float).to_dict()
safety_stock      = df_config["Safety_Stock"].astype(float).to_dict()
minimum_order     = df_config["Min_Order"].astype(float).to_dict()
maximum_order     = df_config["Max_Order"].astype(float).to_dict()
container_capacity= df_config["Container_Cap"].astype(float).to_dict()
shipping_cost     = df_config["Shipping_Cost"].astype(float).to_dict()
purchase_cost     = df_config["Purchase_Cost"].astype(float).to_dict()
holding_cost      = df_config["Holding_Cost"].astype(float).to_dict()
warehouse_space   = df_config["Warehouse_Space"].astype(float).to_dict()

In [ ]:
sales_program = pd.read_excel(
    excel_file,
    sheet_name="Sales_Program",
    index_col="Material",
)

# Remove the reserved/empty programme rows
sales_program = sales_program.dropna(axis=0, how="any")

price_tiers = {}

# Material is the index and may appear multiple times
for material, rows in sales_program.groupby(level=0, sort=False):

    # No need to sort since macros handle this
    price_tiers[material] = {}

    for number, row in enumerate(
        rows.itertuples(index=False),
        start=0,
    ):
        tier = f"T{number}"

        price_tiers[material][tier] = {
            "lower": float(row.start_range)
            * maximum_order[material],

            "upper": float(row.end_range)
            * maximum_order[material],

            "discount": float(row.discount),
        }

In [ ]:

print(f"Successfully imported {len(products)} products, {len(materials)} materials, {warehouse_capacity}m^2 Warehouse and {num_days} planning days!")

**Input Safeguards**

In [ ]:
validation_error = validate_input_safeguards(
    days=days,
    materials=materials,
    products=products,
    requirement=requirement,
    warehouse_space=warehouse_space,
    warehouse_capacity=warehouse_capacity,
    production_due=production_due,
    bom=bom,
    lead_time=lead_time,
    safety_stock=safety_stock,
    initial_inventory=initial_inventory,
)

# -----------------------------------------------------------------------------
# Stop before creating the PuLP model
# -----------------------------------------------------------------------------

if validation_error is not None:

    print("\nINPUT VALIDATION FAILED")
    print(validation_error)
    print("\nModel construction stopped.")

    raise RuntimeError(
        "Material-planning input validation failed."
    )

**Order constraints**

In [ ]:
order_days = {
    m: [
        d for d in days
        if d + lead_time[m] <= max(days)
    ]
    for m in materials
}

order_keys = [
    (m, d)
    for m in materials
    for d in order_days[m]
]

In [ ]:
model = pl.LpProblem(
    "Material_Planning",
    pl.LpMinimize,
)


order_qty = pl.LpVariable.dicts(
    "OrderQty",
    order_keys,
    lowBound=0,
    cat="Continuous",
)

order_placed = pl.LpVariable.dicts(
    "OrderPlaced",
    order_keys,
    cat="Binary",
)

containers = pl.LpVariable.dicts(
    "Containers",
    order_keys,
    lowBound=0,
    cat="Integer",
)

inventory = pl.LpVariable.dicts(
    "Inventory",
    [(m, d) for m in materials for d in days],
    lowBound=0,
    cat="Continuous",
)

In [ ]:
tier_keys = [
    (m, d, tier)
    for m, d in order_keys
    for tier in price_tiers[m]
]

tier_selected = pl.LpVariable.dicts(
    "TierSelected",
    tier_keys,
    cat="Binary",
)

tier_qty = pl.LpVariable.dicts(
    "TierQty",
    tier_keys,
    lowBound=0,
    cat="Continuous",
)

In [ ]:
# Calculate individual cost components
purchasing_cost = pl.lpSum(
    purchase_cost[m] * (1 - price_tiers[m][tier]["discount"]) * tier_qty[m, d, tier]
    for m, d, tier in tier_keys
)

shipping_cost_total = pl.lpSum(
    shipping_cost[m] * containers[m, d]
    for m, d in order_keys
)

holding_cost_total = pl.lpSum(
    holding_cost[m] * inventory[m, d]
    for m in materials
    for d in days
)

# Set objective function
model += purchasing_cost + shipping_cost_total + holding_cost_total, "Total_Cost"

In [ ]:
for m, d in order_keys:

    tiers = list(price_tiers[m])

    # Exactly one tier if an order is placed; zero tiers otherwise
    model += (
        pl.lpSum(
            tier_selected[m, d, tier]
            for tier in tiers
        )
        == order_placed[m, d]
    )

    # Connect tier quantities to the ordinary order quantity
    model += (
        order_qty[m, d]
        == pl.lpSum(
            tier_qty[m, d, tier]
            for tier in tiers
        )
    )

    # Enforce the selected tier's quantity range
    for tier in tiers:
        model += (
            tier_qty[m, d, tier]
            >= price_tiers[m][tier]["lower"]
            * tier_selected[m, d, tier]
        )

        model += (
            tier_qty[m, d, tier]
            <= price_tiers[m][tier]["upper"]
            * tier_selected[m, d, tier]
        )

    # If order_placed = 1, quantity must reach the MOQ
    model += (
        order_qty[m, d]
        >= minimum_order[m] * order_placed[m, d]
    )

    # If order_placed = 0, this forces order quantity to zero
    model += (
        order_qty[m, d]
        <= maximum_order[m] * order_placed[m, d]
    )

    # Container based constraints:
    model += (
        order_qty[m, d]
        <= container_capacity[m] * containers[m, d]
    )

    model += (
        containers[m, d]
        <= ceil(
            maximum_order[m] / container_capacity[m]
        ) * order_placed[m, d]
    )

**Inventory constraints**

In [ ]:
for d in days:

    opening_inventory = {}

    for m in materials:

        if d == 1:
            previous = initial_inventory[m]
        else:
            previous = inventory[m, d - 1]

        # An order arrives after its material-specific lead time:
        arrivals = pl.lpSum(
            order_qty[m, order_day]
            for order_day in order_days[m]
            if order_day + lead_time[m] == d
        )

        # Calculate opening and ending inventory:
        opening_inventory[m] = previous + arrivals

        model += (
            inventory[m, d]
            == opening_inventory[m] - requirement[m, d]
        ), f"InventoryBalance_{m}_{d}"

        model += (
            inventory[m, d] >= safety_stock[m]
        ), f"SafetyStock_{m}_{d}"

    # Warehouse capacity constraint: the total space used by all materials must not exceed the warehouse capacity
    model += (
        pl.lpSum(
            warehouse_space[m] * opening_inventory[m]
            for m in materials
        )
        <= warehouse_capacity
    ), f"Warehouse_{d}"

**Simple Description**

In [ ]:
num_binary = sum(1 for v in model.variables()
                if v.cat == "Integer" and v.lowBound == 0 and v.upBound == 1
)
num_integer = sum(1 for v in model.variables()
                if v.cat == "Integer" and not (v.lowBound == 0 and v.upBound == 1)
)
num_continuous = sum(1 for v in model.variables() if v.cat == "Continuous")

print(f"Total Variables:   {len(model.variables())}")
print(f"  • Continuous:   {num_continuous}")
print(f"  • General Int:  {num_integer}")
print(f"  • Binary (0/1): {num_binary}")
print(f"Total Constraints: {len(model.constraints)}")

**Solve & Print**

In [ ]:
model.solve(pl.HiGHS(gapRel=0.005, timeLimit=300, msg=True))

print("Status:", pl.LpStatus[model.status])
print(
    "Total cost:",
    round(pl.value(model.objective), 2),
)

**Order Plan**

In [ ]:
order_records = []

for m, d in order_keys:

    quantity = order_qty[m, d].value()

    if quantity is not None and quantity > 0.0001:

        chosen_tier = next(
            tier
            for tier in price_tiers[m]
            if (tier_selected[m, d, tier].value() or 0) > 0.5
        )

        tier_info = price_tiers[m][chosen_tier]
        discount = tier_info["discount"]
        discounted_price = purchase_cost[m] * (1 - discount)
        purchase_subtotal = quantity * discounted_price

        order_records.append(
            {
                "Material": m,
                "Order Day": d,
                "Arrival Day": d + lead_time[m],
                "Order Quantity": quantity,
                "Containers": int(round(containers[m, d].value())),
                "Tier": f"{chosen_tier}/{len(price_tiers[m]) -1}",
                "Tier Lower": tier_info["lower"],
                "Tier Upper": tier_info["upper"],
                "Discount": discount,
                "Original Unit Price": purchase_cost[m],
                "Discounted Unit Price": discounted_price,
                "Purchase Subtotal": purchase_subtotal,
            }
        )

order_plan = pd.DataFrame(order_records)

# Preserve the material order from Master_Lists
order_plan["Material"] = pd.Categorical(
    order_plan["Material"],
    categories=materials,
    ordered=True,
)

In [ ]:
order_material_first = (
    order_plan
    .sort_values(["Material", "Order Day"])
    .set_index(["Material", "Order Day"])
)

display(Markdown("### Order plan — material first"))
display(
    style_grouped_table(
        order_material_first,
        order_format,
    )
)

order_day_first = (
    order_plan
    .sort_values(["Order Day", "Material"])
    .set_index(["Order Day", "Material"])
)

display(Markdown("### Order plan — day first"))
display(
    style_grouped_table(
        order_day_first,
        order_format,
    )
)

**Print inventory only on production dates**

In [ ]:
inventory_records = []

for d in days:

    production_happens = any(
        requirement[m, d] > 0
        for m in materials
    )

    if production_happens:

        for m in materials:

            # Inventory carried from the previous day
            previous_inventory = (
                initial_inventory[m]
                if d == 1
                else inventory[m, d - 1].value()
            )

            # Orders arriving on this production day
            arrival_order_days = [
                order_day
                for order_day in order_days[m]
                if order_day + lead_time[m] == d
                and (order_qty[m, order_day].value() or 0) > 0.0001
            ]

            arrivals = sum(
                order_qty[m, order_day].value() or 0
                for order_day in arrival_order_days
            )

            # Actual opening inventory after receiving arrivals
            starting_inventory = previous_inventory + arrivals
            ending_inventory = inventory[m, d].value()

            # Convert material quantity into warehouse occupation
            starting_occupation = (
                starting_inventory * warehouse_space[m]
            )

            ending_occupation = (
                ending_inventory * warehouse_space[m]
            )

            inventory_records.append(
                {
                    "Day": d,
                    "Material": m,
                    "Arrivals": arrivals,
                    "Arrival From Order Day": (
                        ", ".join(map(str, arrival_order_days))
                        if arrival_order_days
                        else "-"
                    ),
                    "Production Requirement": requirement[m, d],
                    "Starting Inventory": starting_inventory,
                    "Ending Inventory": ending_inventory,
                    "Space per Unit": warehouse_space[m],
                    "Starting Occupation (m²)": starting_occupation,
                    "Ending Occupation (m²)": ending_occupation,
                }
            )

df_inventory = pd.DataFrame(inventory_records)

# Preserve the Master_Lists material order
df_inventory["Material"] = pd.Categorical(
    df_inventory["Material"],
    categories=materials,
    ordered=True,
)

In [ ]:
inventory_material_first = (
    df_inventory
    .sort_values(["Material", "Day"])
    .set_index(["Material", "Day"])
)

display(Markdown("### Inventory and production — material first"))
display(
    style_grouped_table(
        inventory_material_first,
        inventory_format,
    )
)

inventory_day_first = (
    df_inventory
    .sort_values(["Day", "Material"])
    .set_index(["Day", "Material"])
)

display(Markdown("### Inventory and production — day first"))
display(
    style_grouped_table(
        inventory_day_first,
        inventory_format,
    )
)

**Check binding constraints**

In [ ]:
warehouse_data = []

for d in days:
    # Total space consumed by ending inventory on day d
    used_space = sum(
        inventory[m, d].value() * warehouse_space[m] for m in materials
    )

    # Room left in the warehouse
    remaining_space = warehouse_capacity - used_space
    pct_free = (remaining_space / warehouse_capacity) * 100

    warehouse_data.append(
        {
            "Day": f"Day {d}",
            "Used Space": f"{used_space:.2f}",
            "Remaining Space": f"{remaining_space:.2f}",
            "Total Capacity": f"{warehouse_capacity:.2f}",
            "% Free": f"{pct_free:.1f}%",
        }
    )

df_warehouse = pd.DataFrame(warehouse_data).set_index("Day")
display(df_warehouse)

In [ ]:
print("\nWAREHOUSE BOTTLENECK CHECK")

for d in days:

    constraint = model.constraints[f"Warehouse_{d}"]

    print(
        f"Day {d}: "
        f"slack = {constraint.slack:.2f}, "
        f"binding = {abs(constraint.slack) <= 1e-6}"
    )